
# Exploratory Data Analysis: Real Belgian Grid Data (Elia Open Data)

Before any model was trained (Components 2-4 of this project), the raw data itself needs
to be understood: how much of it is there, what does it actually look like over time, how
predictable is each series on its face, and how good is the grid operator's OWN forecast
already -- since that's the real bar Component 3's models have to clear, not zero.

This notebook is entirely descriptive: no models are trained or loaded here. Every number
and plot below comes straight from `data/processed/grid_merged.csv`, the output of
`src/forecasting/data_loader.py` + `data_loader.py`'s merge step (see that file's own
docstring for the two real data-quality issues it already handles: wind's small negative
DST/upscaling noise, clipped to 0; solar's 3-level Belgian reporting hierarchy, which has
to be filtered down to national totals before summing, or generation gets multiply
counted).


In [ ]:
import os, sys

def find_project_root(start):
    '''Walk up from `start` until a directory containing both 'src' and 'data' is
    found. Safe to re-run: once cwd IS the project root, it's found immediately.'''
    path = os.path.abspath(start)
    for _ in range(4):
        if os.path.isdir(os.path.join(path, "src")) and os.path.isdir(os.path.join(path, "data")):
            return path
        path = os.path.dirname(path)
    raise RuntimeError(
        "Could not locate the project root (a folder containing both 'src' and 'data') "
        "within 4 levels above the current directory. Run this notebook from inside "
        "smart-grid-rl/notebooks/, or adjust find_project_root's search depth."
    )

PROJECT_ROOT = find_project_root(os.getcwd())
os.chdir(PROJECT_ROOT)
if os.path.join(PROJECT_ROOT, "src") not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("Project root:", PROJECT_ROOT)

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.1f}")


## 1. Overview: how much data, over what period, and how clean is it

A quick sanity check before anything else -- gaps or unexpected nulls here would silently
corrupt every downstream lag/rolling feature in `build_dataset.py`.


In [ ]:
merged = pd.read_csv(os.path.join("data", "processed", "grid_merged.csv"), parse_dates=["timestamp"])

print(f"{len(merged):,} rows")
print(f"Range: {merged['timestamp'].min()} -> {merged['timestamp'].max()} "
      f"({(merged['timestamp'].max() - merged['timestamp'].min()).days} days)")

expected_rows = int((merged["timestamp"].max() - merged["timestamp"].min()).total_seconds() / (15 * 60)) + 1
print(f"Expected rows at a perfect 15-min cadence: {expected_rows:,} "
      f"(actual: {len(merged):,}, missing: {expected_rows - len(merged):,})")

print("\nNull counts per column:")
print(merged.isna().sum())

merged.head()


## 2. Summary statistics

Load, wind, and solar are three physically different processes and it shows immediately
in these numbers: load is a demand series with a comparatively tight, unimodal spread;
solar is bimodal in the crudest sense (a hard floor at exactly 0 MW every night, then a
skewed daytime distribution); wind sits somewhere in between -- always-on but highly
variable.


In [ ]:
summary = merged[["load_mw", "wind_mw", "solar_mw"]].describe().T
summary["zero_pct"] = [
    (merged[col] == 0).mean() * 100 for col in ["load_mw", "wind_mw", "solar_mw"]
]
summary


That `zero_pct` column is the single most important number in this table for anyone
building a forecaster for these series: solar is exactly 0 MW a large fraction of the
time (every single night, by physical necessity), which is exactly why
`lstm_model.py`/`transformer_model.py`/`probabilistic.py` all clip negative predictions
for solar (and wind) to 0 -- a model with no awareness of this hard floor will happily
predict small negative numbers right around it.



## 3. Raw time series

The full series, plus the first 7 days zoomed in -- the zoomed view is where load's daily
commute-driven double-hump and solar's clean daytime bump are actually visible; on the
full-range plot they blur into a solid band.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, col, color in zip(axes, ["load_mw", "wind_mw", "solar_mw"], ["tab:blue", "tab:green", "tab:orange"]):
    ax.plot(merged["timestamp"], merged[col], color=color, linewidth=0.7)
    ax.set_ylabel(col)
    ax.grid(alpha=0.3)
axes[0].set_title("Full series")
plt.tight_layout()
plt.show()

In [ ]:
one_week = merged[merged["timestamp"] < merged["timestamp"].min() + pd.Timedelta(days=7)]

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, col, color in zip(axes, ["load_mw", "wind_mw", "solar_mw"], ["tab:blue", "tab:green", "tab:orange"]):
    ax.plot(one_week["timestamp"], one_week[col], color=color, linewidth=1.2)
    ax.set_ylabel(col)
    ax.grid(alpha=0.3)
axes[0].set_title("First 7 days")
plt.tight_layout()
plt.show()


## 4. Diurnal profile (average by hour of day)

The shaded band is +/- 1 standard deviation across all days in the dataset at that hour --
a narrow band means that hour behaves almost the same every day; a wide one means it
varies a lot day-to-day even at the same clock time.


In [ ]:
merged["hour"] = merged["timestamp"].dt.hour + merged["timestamp"].dt.minute / 60.0

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ["load_mw", "wind_mw", "solar_mw"], ["tab:blue", "tab:green", "tab:orange"]):
    by_hour = merged.groupby("hour")[col].agg(["mean", "std"])
    ax.plot(by_hour.index, by_hour["mean"], color=color)
    ax.fill_between(by_hour.index, by_hour["mean"] - by_hour["std"], by_hour["mean"] + by_hour["std"],
                     color=color, alpha=0.2)
    ax.set_title(col)
    ax.set_xlabel("hour of day")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Day-of-week profile

Load is the series most likely to show a real weekday/weekend split (offices, factories,
retail all run on a weekly schedule); wind and solar are weather- and sun-angle-driven and
have no physical reason to know what day of the week it is, so any wiggle here is noise,
not signal -- worth checking precisely because it's a good sanity check on that assumption
rather than an expected finding.


In [ ]:
merged["dow"] = merged["timestamp"].dt.dayofweek  # Monday=0
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ["load_mw", "wind_mw", "solar_mw"], ["tab:blue", "tab:green", "tab:orange"]):
    by_dow = merged.groupby("dow")[col].mean()
    ax.bar(dow_labels, by_dow.values, color=color)
    ax.set_title(col)
    ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()


## 6. Distributions

Histograms of the raw values -- load's is the closest to a single bell shape (a demand
series with a floor and ceiling set by the grid's own scale); solar's spike at 0 dwarfs
everything else on a shared y-axis, so it gets its own subplot with an independent scale.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ["load_mw", "wind_mw", "solar_mw"], ["tab:blue", "tab:green", "tab:orange"]):
    ax.hist(merged[col], bins=40, color=color, edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel("MW")
plt.tight_layout()
plt.show()


## 7. Correlation between series

Simultaneous-timestamp correlation -- a reminder of why `build_dataset.py` deliberately
does NOT use one target's same-timestamp measurement as a feature for another (see that
file's own docstring): today's actual wind_mw is a measurement, not a forecast, so using
it to help predict today's actual load_mw would be leaking a same-instant reading into
what's supposed to be a forward-looking prediction problem.


In [ ]:
corr = merged[["load_mw", "wind_mw", "solar_mw"]].corr()
corr


## 8. How good is Elia's own day-ahead load forecast?

This is the real, already-deployed baseline Component 3's own load forecasters are
implicitly competing with (see `src/forecasting/evaluate_vs_elia.py`) -- not a strawman.
Elia is a grid operator with access to weather forecasts, historical patterns, and
decades of institutional forecasting practice, so beating this bar (rather than just
beating a naive persistence baseline) is the more meaningful test.


In [ ]:
has_forecast = merged["elia_dayahead_forecast_load_mw"].notna()
print(f"Rows with an Elia day-ahead forecast available: {has_forecast.sum():,} / {len(merged):,}")

if has_forecast.sum() > 0:
    elia_errors = (merged.loc[has_forecast, "elia_dayahead_forecast_load_mw"]
                   - merged.loc[has_forecast, "load_mw"])
    print(f"Elia day-ahead forecast error: MAE = {elia_errors.abs().mean():.1f} MW, "
          f"RMSE = {np.sqrt((elia_errors ** 2).mean()):.1f} MW")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(elia_errors, bins=40, color="tab:red", edgecolor="white")
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title("Elia day-ahead forecast error (forecast - actual)")
    ax.set_xlabel("MW")
    plt.tight_layout()
    plt.show()
else:
    print("No Elia day-ahead forecast values in this slice of data -- nothing to compare.")


## Takeaways

- Solar's exact-0 floor at night and wind's constant availability but high variability are
  real physical properties, not artifacts -- they're why Component 3's models clip
  negative predictions to 0 for both, and why wind is consistently the hardest of the
  three targets to forecast well (see `forecasting_theory.ipynb`'s real MAE/RMSE
  comparison table).
- Load's clear diurnal shape (and comparatively small day-of-week effect in this
  particular slice) is what makes the naive persistence ("same hour yesterday") baseline
  unusually strong for load -- any model has to actually add value beyond that pattern,
  not just learn to reproduce it.
- Elia's own day-ahead forecast error above is the bar this project's forecasters are
  really being measured against; `evaluate_vs_elia.py` and `forecasting_theory.ipynb`
  report the head-to-head result.

**A note on the numbers actually shown when you run this**: this notebook is fully live --
every number and plot above regenerates from whatever is currently in
`data/processed/grid_merged.csv`. The date range and row counts you see depend entirely on
how much real Elia data has been pulled and processed on your machine; if that's a
different window than whoever else runs this notebook, the specific numbers (though not
the qualitative patterns) will differ.
